## 加载与整合数据

In [ ]:
%reset -f
import numpy as np
import anndata as ad
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import os

display(os.getcwd())
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)
sc.settings.verbosity = 3

# 筛选对应的主要细胞类型
chosen_celltype = f":/选择的主要细胞类型"
adata = sc.read(f":/已经准备好的anndata文件路径")
# 提取指定类型对应的大类细胞
adata = adata[adata.obs.loc[:, "major_celltype"].eq(chosen_celltype)].copy()
# 清理无用数据
adata.obsm.clear()
# 使用归一化后的对数表达；scVI 单独读取 counts layer
adata.X = adata.layers['lognorm']
# 保存
adata.write(f"data/{config['project_code']}_{chosen_celltype}.h5ad")
# 展示细胞概况
adata
# 生成时替换整个列表为数值列表
resolutions = [":/挑选合适的聚类分辨率参数"]
cluster_prefix = chosen_celltype + "_"

In [ ]:
# harmony
import scanpy.external as sce

# harmony
sc.tl.pca(adata, n_comps=50)
sce.pp.harmony_integrate(adata=adata, key=['sample'],
                         basis='X_pca',
                         max_iter_harmony = 20,
                         theta = None,
                         lamb = None,
                         sigma = 0.1, 
                         nclust = None,
                         tau = 0,
                         block_size = 0.05, 
                         max_iter_kmeans = 20,
                         epsilon_cluster = 1e-5,
                         epsilon_harmony = 1e-4, 
                         random_state=42,
                         adjusted_basis='X_pca_harmony')


In [ ]:
# scVI
# 12m11.8s
import scvi
import numpy as np

scvi.settings.seed = 12345
# scVI去除批次效应
scvi.model.SCVI.setup_anndata(
    adata, 
    layer="counts", 
    batch_key='sample',
)
model = scvi.model.SCVI(adata)
model.train(accelerator="gpu",early_stopping=True, enable_progress_bar=True,
            batch_size = 256,max_epochs=100)
adata.obsm['X_scVI'] = model.get_latent_representation(adata).astype(np.float32)


In [ ]:
# 两种整合均已计算；默认使用 Harmony，效果不佳时改为 "X_scVI"
# 切换后从本框开始重跑聚类及后续分析，无需重新计算整合
use_rep = "X_pca_harmony"
sc.pp.neighbors(adata=adata, use_rep=use_rep)
sc.tl.umap(adata=adata)
sc.tl.tsne(adata=adata, use_rep=use_rep)

## leiden聚类

In [ ]:
for res in resolutions:
    sc.tl.leiden(adata, flavor="igraph", directed=False, resolution=res, key_added=f"leiden_res{res}")
# 根据分辨率列表生成聚类列名
map_list = [f"leiden_res{res}" for res in resolutions] 

# 指定网格布局
n_cols = min(3, len(map_list))
n_rows = (len(map_list) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 7, n_rows * 4), squeeze=False)

# 将 axes 转换为一维数组以便迭代（如果行列数 > 1）
axes_flat = axes.flatten()

# 为每个 color 绘制 UMAP 图
for i, color in enumerate(map_list):
    if i < n_rows * n_cols:  # 确保不超出子图数量
        ax = axes_flat[i]
        sc.pl.umap(adata, color=color, ax=ax, show=False, title=color)
    else:
        break  # 如果 map_list 比子图多，停止绘制

# 隐藏多余的子图（如果 map_list 不足以填满网格）
for i in range(len(map_list), n_rows * n_cols):
    axes_flat[i].axis("off")

# 调整布局以避免重叠
plt.tight_layout()
plt.savefig(f"figures/{chosen_celltype}_leiden_result.pdf")

In [ ]:
adata_selected = adata.copy()
# 选择聚类结果
cluster_key_chosen = ":/根据聚类图从map_list中选择一个聚类列名"
# 亚群命名
adata_selected.obs.loc[:, "sub_celltype"] = (cluster_prefix + adata_selected.obs.loc[:, cluster_key_chosen].astype(str)).astype('category')
# 数据保存
adata_selected.write(f"data/{config['project_code']}_{chosen_celltype}.h5ad")
adata_selected

## 生物学注释亚群

In [ ]:
%reset -f
import numpy as np
import anndata as ad
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import os

display(os.getcwd())
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)
sc.settings.verbosity = 3

import warnings
warnings.filterwarnings('ignore')

chosen_celltype = f":/选择的主要细胞类型"
tag = f":/本次分析的标签"
adata = sc.read(f"data/{config['project_code']}_{chosen_celltype}.h5ad")
adata

### 差异分析

In [ ]:
sc.tl.rank_genes_groups(adata, 
                        groupby="sub_celltype", 
                        groups='all',
                        reference='rest',
                        method="wilcoxon", 
                        use_raw=False, 
                        layer="lognorm",
                        pts=True)
# 遍历所有细胞亚群
for g in adata.uns['rank_genes_groups']["names"].dtype.names:
    # 筛选差异分析结果
    res = sc.get.rank_genes_groups_df(adata, group=g)
    # 按关键字排序差异分析结果
    res = res.sort_values(
        by=["logfoldchanges", "pct_nz_group", "pvals_adj"],
        ascending=[False, False, True]
    )
    # 过滤没有统计学意义的结果
    mask = (res['logfoldchanges'].abs() > 1) & (res['pvals_adj'] < 0.1)
    # 保存所有有统计学意义的差异分析结果
    res.loc[mask, :].to_csv(f"result/DEG_by_{chosen_celltype}_subcluster/deg_of_{g}_{tag}.tsv", index=None, sep="\t")
    # 保存所有有统计学意义+logfc top100的差异分析结果
    res.sort_values(by="logfoldchanges", ascending=False).loc[mask, :].head(100) \
        .to_csv(f"result/DEG_by_{chosen_celltype}_subcluster/top100_deg_of_{g}_{tag}.tsv", index=None, sep="\t")

In [ ]:
# 绘制圆点热图，指定排序后的分群
sc.pl.rank_genes_groups_dotplot(
    adata,
    n_genes=4,                           # 每群显示 top 4 基因
    values_to_plot="logfoldchanges",     # 显示 log fold change
    min_logfoldchange=1,                 # 最小 log fold change 阈值
    vmax=2,                              # 颜色最大值
    vmin=-2,                             # 颜色最小值
    cmap="coolwarm",                       # 颜色方案
    dendrogram=False,                    # 这一项不加的话默认的纵轴排序顺序是按聚类结果远近排序的
    # swap_axes=True,
    save=f"{chosen_celltype}_subcluster/rank_genes_groups_dotplot.pdf" # 前面有figures/dotplot_前缀
)

### 检查是否需要合并或删除亚群

根据本次差异分析表和聚类图设置下一框的 `merge_map` 和 `exclude_clusters`；默认均为空，不修改亚群。合并映射为原群名到最终群名的一次映射，删除列表使用合并后的群名。不要沿用其他项目的判断。


In [ ]:
# 根据当前数据的证据填写；默认不合并、不删除
merge_map = {}
exclude_clusters = []
subclusters_changed = bool(merge_map or exclude_clusters)
if subclusters_changed:
    labels = adata.obs["sub_celltype"].astype(str)
    adata.obs["sub_celltype"] = labels.map(lambda label: merge_map.get(label, label)).astype("category")
    adata = adata[~adata.obs["sub_celltype"].isin(exclude_clusters)].copy()
    adata.obs["sub_celltype"] = adata.obs["sub_celltype"].cat.remove_unused_categories()


In [ ]:
adata.obs["sub_celltype"].value_counts()

仅在合并或删除亚群后执行下一节。重新分析直接使用修改后的 `adata`。清理差异表时只删除本次 `tag` 的旧导出表，保留目录和其他分析结果。


### 对最终分群再次差异分析

In [ ]:
if subclusters_changed:
    # 仅清理当前分析标签的旧差异表，避免已删除亚群的结果残留
    from pathlib import Path
    deg_dir = Path(f"result/DEG_by_{chosen_celltype}_subcluster")
    for old_table in deg_dir.iterdir():
        if (old_table.is_file()
                and old_table.name.startswith(("deg_of_", "top100_deg_of_"))
                and old_table.name.endswith(f"_{tag}.tsv")):
            old_table.unlink()
    sc.tl.rank_genes_groups(adata, 
                            groupby="sub_celltype", 
                            groups='all',
                            reference='rest',
                            method="wilcoxon", 
                            use_raw=False, 
                            layer="lognorm",
                            pts=True)
    # 遍历所有细胞亚群
    for g in adata.uns['rank_genes_groups']["names"].dtype.names:
        # 筛选差异分析结果
        res = sc.get.rank_genes_groups_df(adata, group=g)
        # 按关键字排序差异分析结果
        res = res.sort_values(
            by=["logfoldchanges", "pct_nz_group", "pvals_adj"],
            ascending=[False, False, True]
        )
        # 过滤没有统计学意义的结果
        mask = (res['logfoldchanges'].abs() > 1) & (res['pvals_adj'] < 0.1)
        # 保存所有有统计学意义的差异分析结果
        res.loc[mask, :].to_csv(f"result/DEG_by_{chosen_celltype}_subcluster/deg_of_{g}_{tag}.tsv", index=None, sep="\t")
        # 保存所有有统计学意义+logfc top100的差异分析结果
        res.sort_values(by="logfoldchanges", ascending=False).loc[mask, :].head(100) \
            .to_csv(f"result/DEG_by_{chosen_celltype}_subcluster/top100_deg_of_{g}_{tag}.tsv", index=None, sep="\t")


In [ ]:
if subclusters_changed:
    # 绘制圆点热图，指定排序后的分群
    sc.pl.rank_genes_groups_dotplot(
        adata,
        n_genes=4,                           # 每群显示 top 4 基因
        values_to_plot="logfoldchanges",     # 显示 log fold change
        min_logfoldchange=1,                 # 最小 log fold change 阈值
        vmax=2,                              # 颜色最大值
        vmin=-2,                             # 颜色最小值
        cmap="coolwarm",                       # 颜色方案
        dendrogram=False,                    # 这一项不加的话默认的纵轴排序顺序是按聚类结果远近排序的
        # swap_axes=True,
        # save=f"{chosen_celltype}_subcluster/rank_genes_groups_dotplot.pdf" # 前面有figures/dotplot_前缀
    )


<div style="
    background-color: #f5f0ff;
    border-left: 5px solid #7c3aed;
    padding: 12px 16px;
    margin: 16px 0;
    border-radius: 5px;
">
<strong>⏸ AGENT CHECKPOINT — PAUSE EXECUTION</strong><br>
暂停执行后续代码。首先读取并分析
<code>result/DEG_by_{chosen_celltype}_subcluster</code>，
根据 marker gene 判断每个 cluster 的细胞类型。
完成细胞类型判断后，再修改并执行下方 annotation 代码中的`genes`和`cluster2annotation`
</div>

### 选取marker基因

In [ ]:
# 根据上面分析结果挑选差异最明显且表达占比也相对较高的marker基因
genes = {":/当前亚群名": [":/该亚群的marker基因"]}

# 绘制圆点热图，指定排序后的分群
sc.pl.rank_genes_groups_dotplot(
    adata,
    groupby="sub_celltype",
    var_names=genes,                 
    values_to_plot="logfoldchanges",     # 显示 log fold change
    min_logfoldchange=1,                 # 最小 log fold change 阈值
    vmax=2,                              # 颜色最大值
    vmin=-2,                             # 颜色最小值
    cmap="coolwarm",                       # 颜色方案
    dendrogram=False,                    # 这一项不加的话默认的纵轴排序顺序是按聚类结果远近排序的
    # swap_axes=True,
    save=f"{chosen_celltype}_subcluster/rank_genes_groups_dotplot.pdf" # 前面有figures/dotplot_前缀
)

根据最终亚群填写完整的 `cluster2annotation`，执行下一框生成 `_annot.h5ad` 后再进入可视化。


In [ ]:
# 根据本次分析填写所有最终亚群的生物学注释
cluster2annotation = {":/当前亚群名": ":/该亚群的生物学注释"}
adata_selected = adata.copy()
adata_selected.obs['sub_celltype'] = adata_selected.obs['sub_celltype'].map(cluster2annotation).astype('category')

# 1) 删除 obs 中所有列名包含 'leiden' 的项
leiden_obs_keys = [k for k in adata_selected.obs.columns if "leiden" in k]
adata_selected.obs.drop(columns=leiden_obs_keys, inplace=True)

print("Removed obs keys:", leiden_obs_keys)

# 2) 删除 uns 中所有 key 包含 'leiden' 的项
leiden_uns_keys = [k for k in list(adata_selected.uns.keys()) if "leiden" in k]
for k in leiden_uns_keys:
    del adata_selected.uns[k]

print("Removed uns keys:", leiden_uns_keys)

adata_selected.write(f"data/{config['project_code']}_{chosen_celltype}_annot.h5ad")
adata_selected